# LFS Dataset Extraction
Using Python to extract a subset from the Labour Force Survey microdata. This walkthrough reads data from zipfiles, rather than needing to unzip all the datasets.

## A. Import libraries to work with dataframes and zipfiles


In [1]:
import pandas as pd
import zipfile as zf

In [2]:
OUTPUT_FILE = 'LFS_Ontario_2015-2026.csv'
YEAR_START = 2015
YEAR_END = 2026

# CONDITION is used to filter results.
# eg. 'PROV=35' is interpreted as finding all records where PROV is coded as
CONDITION = 'PROV=35'


## B. Define Functions

### get_year_data(year, condition)
Pull and consolidate a particular year's microdata files. Accepts an optional condition or filter in the form of KEY=VALUE.

*This version assumes the value is a number, but it would be easy to change the check to compare strings.*

For example:
PROV=35

In [6]:
def get_year_data(year,condition=''):
    df_result = pd.DataFrame()
    bStarted = False
    filelist = ''
    key = ''
    value = ''
    test = '' # Condition to check

    # Check if a condition is provided and has an = divider
    if len(condition)>0 and condition.find('=') > -1:
        key,value = condition.split('=')

    # Load the zip file for the year
    fname = f'{str(year)}-CSV.zip'
    with zf.ZipFile(fname,'r') as zip_file:

        # Retrieve a list of all files
        filelist = zip_file.namelist()
        for file in filelist:

            # Check if file matches the 'pubMMYY.csv' format where MM
            # is the month number and YY is the last two digits of the year
            if file.startswith('pub') and file.endswith('.csv'):
                # Attempt to open the file
                try:
                    print(f'filename = {file}')
                    # Open the public microdata file in the zip and add to dataframe
                    with zip_file.open(file) as f:
                        df = pd.read_csv(f)
                        if not bStarted:
                            df_result = df
                            bStarted = True
                        else:
                            df_result = pd.concat([df_result,df],ignore_index=True)
                except:
                    print(f'ERROR: {file} failed')

    # Check if condition exists, then apply it to retrieve just the subset
    if len(condition) > 0:

        # Check if key is a valid column name
        if key in df_result.columns:
            test = df_result[key] == int(value) # Use this for numbers
            # test = df_result[key] == value # Use this version for strings
            df_result = df_result[test]
        else:
            print(f'{key} not found in dataframe')

    return df_result

## C. Run Extraction

Modify this to retrieve data for specific years and/or particular condition.

In [5]:
df_complete = pd.DataFrame()
bFirst = True

for year in range(YEAR_START,YEAR_END+1):
    print(f'Current year: {year}')
    df_data = get_year_data(year,CONDITION)
    if df_complete.empty:
        df_complete = df_data
    else:
        df_complete = pd.concat([df_complete,df_data],ignore_index=True)

df_complete.to_csv(OUTPUT_FILE)

Current year: 2024
filename = pub0124.csv
filename = pub0224.csv
filename = pub0324.csv
filename = pub0424.csv
filename = pub0524.csv
filename = pub0624.csv
filename = pub0724.csv
filename = pub0824.csv
filename = pub0924.csv
filename = pub1024.csv
filename = pub1124.csv
filename = pub1224.csv
Current year: 2025
filename = pub0125.csv
filename = pub0225.csv
filename = pub0325.csv
filename = pub0425.csv
filename = pub0525.csv
filename = pub0625.csv
filename = pub0725.csv
filename = pub0825.csv
filename = pub0925.csv
filename = pub1025.csv
filename = pub1125.csv
filename = pub1225.csv
Current year: 2026
filename = pub0226.csv
filename = pub0126.csv
